# PUF256 FPGA 驗證

獨立 PYNQ-Z2 overlay：640 RO / 320 候選配對 / 選 256 / 每組 5 次 / 80-bit helper。PUF、投票和 ECC 全部在 PL 執行；Python 只做 MMIO 與實驗紀錄。

載入此 overlay 會取代目前 PL 設計。請僅在已借用的板子執行，並確保沒有其他程式同時控制 PL。

此測試沒有 SHA3、身分認證、溫壓掃描或跨晶片唯一性驗證。

In [ ]:
from pathlib import Path
import hashlib, json, platform
root = Path.cwd()
manifest = json.loads((root / 'MANIFEST.json').read_text())
for name, record in manifest['files'].items():
    if name.endswith('.ipynb'): continue  # Execution changes notebook outputs.
    data = (root / name).read_bytes()
    assert len(data) == record['bytes']
    assert hashlib.sha256(data).hexdigest() == record['sha256'], name
print('Upload hashes verified')
print('Python', platform.python_version())
print('Bitstream SHA256', manifest['files']['puf256.bit']['sha256'])


## 完整實驗

以 32 輪 reference + 128 輪 training 篩選 320 組；根據訓練資料的零 fault/tie/flip/weak 與精確最小頻率差比例，選定並凍結 256 組。接著才做 32 輪獨立 holdout，不會因 holdout 失敗而替換配對。

硬體 Registration 只做一次；之後 100 次 Certification、一次共同 abort/reset 後重建、10 次重載相同 bitstream 後重建。重新載入時 bitmap/helper 寄存器會清空，由外部 enrollment.json 再提供。

CSV 包含實驗用 raw PUF 參考資料，僅供專題分析，不應當成正式產品的秘密儲存方案。整段可能需要數分鐘；每 8 輪列印進度。

In [ ]:
from board_validation import run_validation
summary = run_validation(bitfile='puf256.bit', reference_reads=32, training_reads=128,
                         holdout_reads=32, reconstruction_runs=100, reload_runs=10)
assert summary['passed']


## 檢查並匯出結果

每次執行會產生獨立 results_日期時間 資料夾。summary.json 只有全部通過才會產生 passed=true；中斷或錯誤時看 progress.json 與既有 CSV，不能宣稱完成。將該資料夾下載後，以 00_TESTBED/verify_board_results.py 獨立重算篩選、helper 與重建結果。

In [ ]:
folders = sorted(p for p in root.glob('results_*') if p.is_dir())
latest = folders[-1]
print('Latest results:', latest.name)
print((latest / 'summary.json').read_text())
import zipfile
archive = root / (latest.name + '.zip')
with zipfile.ZipFile(str(archive), 'w', zipfile.ZIP_DEFLATED) as bundle:
    for path in sorted(latest.iterdir()):
        if path.is_file(): bundle.write(str(path), latest.name + '/' + path.name)
from IPython.display import FileLink, display
display(FileLink(archive.name))
